# SVV-kamera – YOLO26 trening på Google Colab

**Steg:**
1. Koble til Google Drive
2. Last opp `dataset.zip` til Drive
3. Kjør cellene i rekkefølge
4. Beste vekter lagres til `MyDrive/svv_yolo/best.pt`

**Krav:** Kjør med GPU-akselerasjon: `Rediger → Notatbokinnstillinger → T4 GPU`

In [ ]:
# ── 1. Sjekk GPU ─────────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ Ingen GPU funnet – bytt runtime til T4 GPU')

In [ ]:
# ── 2. Installer pakker ───────────────────────────────────────────────────────
%pip install ultralytics -q
print('✅ ultralytics installert')

In [ ]:
# ── 3. Koble til Google Drive ─────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive montert')

In [ ]:
# ── 4. Pakk ut dataset ────────────────────────────────────────────────────────
# Last opp dataset.zip til Google Drive først:
#   Gå til drive.google.com → Min Disk → legg inn dataset.zip
#
# Lag dataset.zip lokalt (kjør i terminalen din):
#   cd /Users/sondre/svv_kamera/Master-s-Thesis-V26---Sondre-Eirik/Model
#   zip -r dataset.zip dataset/ dataset.yaml

import zipfile, os
from pathlib import Path

ZIP_PATH    = '/content/drive/MyDrive/dataset.zip'   # ← juster hvis du la den i en mappe
EXTRACT_DIR = '/content/svv_dataset'

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(EXTRACT_DIR)

print(f'✅ Dataset pakket ut til {EXTRACT_DIR}')
!find {EXTRACT_DIR} -maxdepth 3 -type d

In [ ]:
# ── 5. Fiks dataset.yaml med korrekte Colab-stier ────────────────────────────
import yaml

YAML_SRC = f'{EXTRACT_DIR}/dataset.yaml'

with open(YAML_SRC) as f:
    cfg = yaml.safe_load(f)

cfg['path']  = f'{EXTRACT_DIR}/dataset'
cfg['train'] = 'images/train'
cfg['val']   = 'images/val'

YAML_OUT = '/content/dataset_colab.yaml'
with open(YAML_OUT, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True)

print('✅ dataset_colab.yaml:')
!cat {YAML_OUT}

In [ ]:
# ── 6. Trening ────────────────────────────────────────────────────────────────
from ultralytics import YOLO

MODEL   = 'yolo26n.pt'   # nano – rask og lett
EPOCHS  = 100
IMGSZ   = 640            # full oppløsning – GPU har nok minne
BATCH   = 16             # T4 har 16 GB – trygt med batch 16

model = YOLO(MODEL)

results = model.train(
    data      = YAML_OUT,
    epochs    = EPOCHS,
    imgsz     = IMGSZ,
    batch     = BATCH,
    device    = 0,          # GPU
    patience  = 20,
    pretrained= True,
    project   = '/content/runs',
    name      = 'svv_kamera',
    exist_ok  = True,
    save      = True,
    plots     = True,
)

print('\n✅ Trening ferdig!')

In [ ]:
# ── 7. Validering ─────────────────────────────────────────────────────────────
metrics = model.val()
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

In [ ]:
# ── 8. Kopier beste vekter til Google Drive ───────────────────────────────────
import shutil

BEST_PT  = '/content/runs/svv_kamera/weights/best.pt'
DRIVE_OUT = '/content/drive/MyDrive/svv_yolo/'

os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copy2(BEST_PT, DRIVE_OUT + 'best.pt')
print(f'✅ best.pt lagret til {DRIVE_OUT}best.pt')

# Kopier også treningsplottene
for f in Path('/content/runs/svv_kamera').glob('*.png'):
    shutil.copy2(f, DRIVE_OUT)
print('✅ Treningsplott kopiert til Drive')

In [ ]:
# ── 9. Last ned best.pt direkte til PC (alternativ til Drive) ─────────────────
from google.colab import files
files.download(BEST_PT)